<a href="https://colab.research.google.com/github/DiegoMartell/Deep-Learning/blob/main/Hands-on/Hands_on_5_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Hands-on 5: Redes Neuronales Convolucionales (CNN)
### Implementación paso a paso con Keras/TensorFlow (Dataset MNIST)

## 1. Fundamentos Teóricos y Matemáticos de las CNN

A diferencia de los Perceptrones Multicapa (MLP) que aplanan una imagen, las **Redes Neuronales Convolucionales (CNN)** analizan las imágenes en su forma original bidimensional para mantener la relación espacial entre los píxeles.

### 1.1 ¿Cómo "ven" las CNN?
Las CNN utilizan **Filtros (o Kernels)** (pequeñas matrices de pesos) que se deslizan a través de la imagen escaneando pequeñas porciones.

* **Primeras capas:** Detectan características simples (bordes, líneas).
* **Capas profundas:** Combinan esos bordes para reconocer formas complejas (un dígito completo).

### 1.2 La Operación de Convolución
Matemáticamente, la convolución entre una imagen $I$ y un filtro $K$ es:
$$S(i, j) = (I * K)(i, j) = \sum_m \sum_n I(i+m, j+n) K(m, n)$$
Se multiplica el filtro por la región de la imagen y se suma el resultado en un solo píxel del mapa de características.

In [ ]:
# Importamos librerías base y herramientas de Keras
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

print("✅ Entorno inicializado.")

## 2. Preparación de los Datos (Dataset MNIST)

El dataset MNIST contiene 70,000 imágenes en escala de grises ($28 \times 28$ píxeles) de dígitos escritos a mano (0-9).

In [ ]:
# 1. Cargar el dataset MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# 2. Reformatear imágenes para agregar el canal de color (escala de grises = 1)
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

# 3. Normalizar píxeles (de 0-255 a 0.0-1.0)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# 4. Codificación One-Hot de etiquetas
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"✅ Datos listos. Forma de X_train: {X_train.shape}")

# Visualizamos los primeros 5 dígitos
plt.figure(figsize=(10, 2))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(X_train[i].reshape(28, 28), cmap='gray')
    plt.title(f'Etiqueta: {y_train[i]}')
    plt.axis('off')
plt.show()

## 3. Construcción del Modelo CNN: Etapa por Etapa

In [ ]:
cnn_model = Sequential()

# ETAPA 1 y 3: CONVOLUCIÓN y RELU
# Escanea con 32 filtros 3x3 y aplica activación ReLU para no-linealidad.
cnn_model.add(Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))

# ETAPA 2: MAX POOLING
# Reduce dimensiones espaciales reteniendo el valor máximo de cada región 2x2.
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))

# Segundo bloque Conv+ReLU+Pool para características abstractas
cnn_model.add(Conv2D(filters=64, kernel_size=(3, 3), activation='relu'))
cnn_model.add(MaxPooling2D(pool_size=(2, 2)))

# ETAPA 4: FLATTENING
# Aplana los mapas de características a un vector 1D.
cnn_model.add(Flatten())

# ETAPA 5: DENSE (Fully Connected)
# Capa clasificadora final.
cnn_model.add(Dense(units=128, activation='relu'))
cnn_model.add(Dropout(0.5)) # Prevención de Overfitting

# Salida: 10 neuronas (dígitos 0-9) con activación Softmax (probabilidades).
cnn_model.add(Dense(units=10, activation='softmax'))

cnn_model.summary()

## 4. Compilación y Entrenamiento

In [ ]:
# Compilamos con optimizador Adam y pérdida de entropía cruzada categórica
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

print("🚀 Iniciando entrenamiento...")
# Entrenamos por 5 épocas
historia = cnn_model.fit(
    X_train, y_train_cat,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 5. ETAPA 6: Evaluation (Evaluación del Modelo)

Evaluamos el modelo contra el conjunto de Prueba (`X_test`) para comprobar su capacidad de generalización.

In [ ]:
test_loss, test_acc = cnn_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"🎯 Exactitud (Accuracy) en datos no vistos: {test_acc * 100:.2f}%")

# Curvas de aprendizaje
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(historia.history['accuracy'], label='Entrenamiento', color='blue')
plt.plot(historia.history['val_accuracy'], label='Validación', color='orange')
plt.title('Curva de Exactitud (Accuracy)')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(historia.history['loss'], label='Entrenamiento', color='blue')
plt.plot(historia.history['val_loss'], label='Validación', color='orange')
plt.title('Curva de Pérdida (Loss)')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. ETAPA 7: Prediction (Predicciones)

In [ ]:
# Predicciones sobre el conjunto de prueba
predicciones_proba = cnn_model.predict(X_test)
predicciones_finales = np.argmax(predicciones_proba, axis=1)

# Mostramos 6 casos aleatorios
indices_aleatorios = np.random.choice(len(X_test), 6, replace=False)

plt.figure(figsize=(12, 5))
for i, idx in enumerate(indices_aleatorios):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_test[idx].reshape(28, 28), cmap='Blues')

    etiqueta_real = y_test[idx]
    prediccion = predicciones_finales[idx]
    probabilidad = np.max(predicciones_proba[idx]) * 100

    color_texto = 'green' if etiqueta_real == prediccion else 'red'
    plt.title(f'Pred: {prediccion} ({probabilidad:.1f}%)\nReal: {etiqueta_real}', color=color_texto, fontweight='bold')
    plt.axis('off')

plt.tight_layout()
plt.show()